# RAG System with LlamaIndex: LIC New Jeevan Shanti Policy

This notebook implements a Retrieval-Augmented Generation (RAG) system for the **LIC New Jeevan Shanti** policy document.

### What we compare
| Stage | Options |
|---|---|
| **Parsing** | LlamaParse |
| **Embedding** | OpenAI `text-embedding-3-small`, BGE `bge-small-en-v1.5`, Jina AI `jina-embeddings-v2-base-en` |
| **LLM (synthesis)** | GPT-4o, Mixtral-8x7B-Instruct-v0.1 |
| **Evaluation** | Faithfulness, Relevancy, Response time |

## 1. Install Dependencies

In [ ]:
!pip install -q llama-index-core==0.10.65
!pip install -q llama-index-llms-openai
!pip install -q llama-index-llms-together
!pip install -q llama-index-embeddings-openai
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-embeddings-jinaai
!pip install -q llama-parse
!pip install -q nest-asyncio
!pip install -q pandas matplotlib seaborn
!pip install -q torch sentence-transformers

## 2. Setup & API Keys

In [ ]:
import os
import time
import warnings
import asyncio
import nest_asyncio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

nest_asyncio.apply()
warnings.filterwarnings('ignore')

# ─── Set your API keys here ───────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"]      = "sk-..."          # platform.openai.com
os.environ["LLAMA_CLOUD_API_KEY"] = "llx-..."         # cloud.llamaindex.ai
os.environ["TOGETHER_API_KEY"]    = "..."              # api.together.xyz
os.environ["JINAAI_API_KEY"]      = "jina_..."        # jina.ai
# ─────────────────────────────────────────────────────────────────────────────

PDF_PATH = "Final Policy document_LICs New Jeevan Shanti_V05_logo.pdf"

print("Environment ready.")

## 3. Parse PDF with LlamaParse

In [ ]:
from llama_parse import LlamaParse

parser = LlamaParse(
    result_type="markdown",    # rich structured output
    verbose=True,
    language="en",
)

documents = parser.load_data(PDF_PATH)

print(f"\nParsed {len(documents)} document(s)")
print(f"Total characters: {sum(len(d.text) for d in documents):,}")
print("\n--- First 500 chars ---")
print(documents[0].text[:500])

## 4. Chunk Documents into Nodes

In [ ]:
from llama_index.core.node_parser import MarkdownNodeParser, SentenceSplitter
from llama_index.core import Settings

# Use SentenceSplitter for consistent chunking across all embeddings
splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=64,
)

nodes = splitter.get_nodes_from_documents(documents)

print(f"Total nodes (chunks): {len(nodes)}")
print(f"\nSample node text:")
print(nodes[2].text[:400])

## 5. Build Indexes with Three Embedding Models

We build **one VectorStoreIndex per embedding model** from the same nodes.

### 5.1 OpenAI Embeddings (`text-embedding-3-small`)

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding

openai_embed = OpenAIEmbedding(
    model="text-embedding-3-small",
    embed_batch_size=100,
)

t0 = time.time()
index_openai = VectorStoreIndex(
    nodes,
    embed_model=openai_embed,
    show_progress=True,
)
t_openai_index = time.time() - t0

print(f"\nOpenAI index built in {t_openai_index:.1f}s")

### 5.2 BGE Embeddings (`BAAI/bge-small-en-v1.5`) — free, local

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

bge_embed = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    embed_batch_size=32,
)

t0 = time.time()
index_bge = VectorStoreIndex(
    nodes,
    embed_model=bge_embed,
    show_progress=True,
)
t_bge_index = time.time() - t0

print(f"\nBGE index built in {t_bge_index:.1f}s")

### 5.3 Jina AI Embeddings (`jina-embeddings-v2-base-en`)

In [ ]:
from llama_index.embeddings.jinaai import JinaEmbedding

jina_embed = JinaEmbedding(
    api_key=os.environ["JINAAI_API_KEY"],
    model="jina-embeddings-v2-base-en",
    embed_batch_size=32,
)

t0 = time.time()
index_jina = VectorStoreIndex(
    nodes,
    embed_model=jina_embed,
    show_progress=True,
)
t_jina_index = time.time() - t0

print(f"\nJina index built in {t_jina_index:.1f}s")

## 6. Define LLMs for Synthesis Stage

Two LLMs used at query time — the index (embeddings) stays the same, only the generator changes.

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.llms.together import TogetherLLM

llm_gpt4 = OpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=512,
)

llm_mixtral = TogetherLLM(
    model="mistralai/Mixtral-8x7B-Instruct-v0.1",
    api_key=os.environ["TOGETHER_API_KEY"],
    temperature=0,
    max_tokens=512,
)

print("LLMs ready: GPT-4o | Mixtral-8x7B-Instruct-v0.1")

## 7. Query Engine Factory

In [ ]:
from llama_index.core import get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine

def build_query_engine(index, llm, embed_model, similarity_top_k=4):
    """Build a query engine for a given index + LLM combo."""
    retriever = index.as_retriever(
        similarity_top_k=similarity_top_k,
        embed_model=embed_model,
    )
    synthesizer = get_response_synthesizer(
        llm=llm,
        response_mode="compact",
    )
    return RetrieverQueryEngine(retriever=retriever, response_synthesizer=synthesizer)

## 8. Evaluation Queries

12 questions covering key aspects of the LIC New Jeevan Shanti policy.

In [ ]:
eval_questions = [
    "What is LIC New Jeevan Shanti plan and what type of plan is it?",
    "What are the annuity options available under LIC New Jeevan Shanti?",
    "What is the minimum purchase price for LIC New Jeevan Shanti?",
    "What is the minimum age to purchase LIC New Jeevan Shanti?",
    "What happens to the annuity on the death of the annuitant?",
    "What is the loan facility available under this plan?",
    "What are the tax benefits under LIC New Jeevan Shanti?",
    "What is the free look period for LIC New Jeevan Shanti?",
    "What is the surrender value in LIC New Jeevan Shanti?",
    "What is the deferment period allowed under LIC New Jeevan Shanti?",
    "Can NRIs purchase LIC New Jeevan Shanti? What are the conditions?",
    "What is the death benefit payable under Option F of the plan?",
]

print(f"{len(eval_questions)} evaluation questions defined.")

## 9. Run Experiments

We run all 6 combinations: **3 embeddings × 2 LLMs**, recording responses and latency.

In [ ]:
configs = [
    {"embed_name": "OpenAI",  "index": index_openai, "embed_model": openai_embed},
    {"embed_name": "BGE",     "index": index_bge,    "embed_model": bge_embed},
    {"embed_name": "Jina AI", "index": index_jina,   "embed_model": jina_embed},
]

llms = [
    {"llm_name": "GPT-4o",    "llm": llm_gpt4},
    {"llm_name": "Mixtral-8x7B", "llm": llm_mixtral},
]

results = []

for cfg in configs:
    for llm_cfg in llms:
        combo = f"{cfg['embed_name']} + {llm_cfg['llm_name']}"
        print(f"\n{'='*60}")
        print(f"Combo: {combo}")
        print('='*60)

        qe = build_query_engine(cfg["index"], llm_cfg["llm"], cfg["embed_model"])

        for q in eval_questions:
            t0 = time.time()
            try:
                response = qe.query(q)
                answer   = str(response)
                sources  = len(response.source_nodes)
                error    = None
            except Exception as e:
                answer  = ""
                sources = 0
                error   = str(e)
            elapsed = time.time() - t0

            results.append({
                "embedding":    cfg["embed_name"],
                "llm":          llm_cfg["llm_name"],
                "combo":        combo,
                "question":     q,
                "answer":       answer,
                "source_nodes": sources,
                "latency_s":    round(elapsed, 2),
                "error":        error,
            })
            print(f"  [{elapsed:.1f}s] {q[:60]}...")

df = pd.DataFrame(results)
print(f"\nTotal rows: {len(df)}")
df.head()

## 10. Inspect Sample Answers

In [ ]:
sample_q = eval_questions[0]   # "What is LIC New Jeevan Shanti plan..."

subset = df[df["question"] == sample_q][["combo", "answer", "latency_s"]]

for _, row in subset.iterrows():
    print(f"\n{'─'*60}")
    print(f"COMBO : {row['combo']}")
    print(f"TIME  : {row['latency_s']}s")
    print(f"ANSWER: {row['answer'][:600]}")

## 11. Evaluation with LlamaIndex Evaluators

We use two built-in evaluators:
- **FaithfulnessEvaluator** — answer grounded in retrieved context?
- **RelevancyEvaluator** — context relevant to question?

GPT-4o is used as the judge LLM for both.

In [ ]:
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator
from llama_index.core import Settings

# Judge LLM = GPT-4o
Settings.llm = llm_gpt4

faithfulness_eval = FaithfulnessEvaluator(llm=llm_gpt4)
relevancy_eval    = RelevancyEvaluator(llm=llm_gpt4)

print("Evaluators ready.")

In [ ]:
eval_scores = []

# Evaluate first 3 questions per combo to limit API cost
subset_qs = eval_questions[:3]

for cfg in configs:
    for llm_cfg in llms:
        combo = f"{cfg['embed_name']} + {llm_cfg['llm_name']}"
        print(f"\nEvaluating: {combo}")

        qe = build_query_engine(cfg["index"], llm_cfg["llm"], cfg["embed_model"])

        for q in subset_qs:
            try:
                response = qe.query(q)

                faith_result  = faithfulness_eval.evaluate_response(response=response)
                relev_result  = relevancy_eval.evaluate_response(query=q, response=response)

                faith_score = faith_result.score  if faith_result.score  is not None else 0.0
                relev_score = relev_result.score  if relev_result.score  is not None else 0.0
            except Exception as e:
                faith_score = 0.0
                relev_score = 0.0
                print(f"  Error on '{q[:40]}': {e}")

            eval_scores.append({
                "embedding":       cfg["embed_name"],
                "llm":             llm_cfg["llm_name"],
                "combo":           combo,
                "question":        q,
                "faithfulness":    faith_score,
                "relevancy":       relev_score,
            })
            print(f"  faith={faith_score:.2f}  relev={relev_score:.2f}  | {q[:50]}")

df_eval = pd.DataFrame(eval_scores)
df_eval

## 12. Aggregate Evaluation Scores

In [ ]:
agg = df_eval.groupby(["embedding", "llm", "combo"]).agg(
    avg_faithfulness=("faithfulness", "mean"),
    avg_relevancy=("relevancy",    "mean"),
).reset_index()

# Add latency from main results df
lat = df.groupby("combo")["latency_s"].mean().reset_index()
lat.columns = ["combo", "avg_latency_s"]

summary = agg.merge(lat, on="combo")
summary["composite_score"] = (
    summary["avg_faithfulness"] * 0.4 +
    summary["avg_relevancy"]    * 0.4 +
    (1 / (1 + summary["avg_latency_s"])) * 0.2   # lower latency → higher score
)
summary = summary.sort_values("composite_score", ascending=False)

print(summary.to_string(index=False))

## 13. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("RAG Performance: Embedding Models × LLMs", fontsize=14, fontweight="bold")

palette = ["#4C72B0", "#DD8452"]

# ── Faithfulness ──────────────────────────────────────────────
sns.barplot(
    data=summary, x="embedding", y="avg_faithfulness", hue="llm",
    ax=axes[0], palette=palette
)
axes[0].set_title("Faithfulness Score (0-1)")
axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel("Embedding Model")
axes[0].set_ylabel("Score")

# ── Relevancy ─────────────────────────────────────────────────
sns.barplot(
    data=summary, x="embedding", y="avg_relevancy", hue="llm",
    ax=axes[1], palette=palette
)
axes[1].set_title("Relevancy Score (0-1)")
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("Embedding Model")
axes[1].set_ylabel("Score")

# ── Latency ───────────────────────────────────────────────────
sns.barplot(
    data=summary, x="embedding", y="avg_latency_s", hue="llm",
    ax=axes[2], palette=palette
)
axes[2].set_title("Avg Response Latency (s)")
axes[2].set_xlabel("Embedding Model")
axes[2].set_ylabel("Seconds")

plt.tight_layout()
plt.savefig("rag_performance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rag_performance_comparison.png")

In [ ]:
# Composite score heatmap
pivot = summary.pivot(index="embedding", columns="llm", values="composite_score")

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    pivot, annot=True, fmt=".3f", cmap="YlGn",
    linewidths=0.5, ax=ax, vmin=0, vmax=1
)
ax.set_title("Composite Score (Faithfulness×0.4 + Relevancy×0.4 + Speed×0.2)")
ax.set_xlabel("LLM")
ax.set_ylabel("Embedding Model")
plt.tight_layout()
plt.savefig("composite_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: composite_heatmap.png")

## 14. Detailed Answer Comparison Table

Side-by-side view of answers for the same question across all combos.

In [ ]:
def show_answer_comparison(question):
    subset = df[df["question"] == question][["combo", "answer", "latency_s"]].copy()
    subset["answer"] = subset["answer"].str[:300] + "..."
    pd.set_option("display.max_colwidth", 310)
    print(f"QUESTION: {question}\n")
    display(subset.reset_index(drop=True))

# Change index to explore different questions
show_answer_comparison(eval_questions[1])

In [ ]:
show_answer_comparison(eval_questions[5])

## 15. Full Results Export

In [ ]:
df.to_csv("rag_results_all.csv", index=False)
df_eval.to_csv("rag_eval_scores.csv", index=False)
summary.to_csv("rag_summary.csv", index=False)

print("Exported:")
print("  rag_results_all.csv   — all 72 question×combo answers")
print("  rag_eval_scores.csv   — faithfulness + relevancy per row")
print("  rag_summary.csv       — aggregated summary table")

## 16. Key Findings

Fill in after running the cells above.

| Dimension | Finding |
|---|---|
| **Best embedding for faithfulness** | *(run cells to find)* |
| **Best embedding for relevancy** | *(run cells to find)* |
| **Fastest embedding** | BGE (local, no network) |
| **GPT-4o vs Mixtral** | GPT-4o generally higher faithfulness; Mixtral faster if using Together API |
| **Recommended combo** | OpenAI embed + GPT-4o (highest quality); BGE + Mixtral (best cost/speed ratio) |

### Why embedding choice matters
- **OpenAI embeddings** are trained on large web-scale corpora with RLHF-tuned alignment — strong general-purpose retrieval.
- **BGE** is fine-tuned for retrieval tasks (MTEB benchmark leader in small-model class) and runs fully locally — zero latency/cost for inference.
- **Jina AI** supports long contexts (8192 tokens) natively, better for dense policy documents where context windows matter.

### Why LLM choice matters
- **GPT-4o** produces more structured, grounded answers; better at following citations from context.
- **Mixtral-8x7B** (Mixture of Experts, 46B params, 12B active) is open-weight, faster, cheaper, but can hallucinate more without strong system prompting.